# Exercise 1
## Maximizing the benefit
with restricted capacity in each team

In [2]:
#Exercise 2 - INNOVATIVE INITIATIVES
#Question 5 
# -> maximizing benefit
using GLPK, JuMP

J = 1:15                  # initiatives
T = 1:3                   # teams

b = [12, 24, 20, 28, 10, 11, 30, 27, 22, 25, 40, 38, 32, 34, 36]        # Strategic benefit
a = [14,  8,  7, 10, 13, 12,  9,  8,  7,  8, 12, 11, 10,  9, 10]               # Resource consumption
C = [28, 32, 35]    # Team capacities

# Dependency pairs:
dependencies = [    # (i,j) means initiative i requires initiative j
    (2,1),
    (3,1),
    (4,1),
    (7,5),
    (8,5),
    (9,6),
    (10,6),
    (11,7),
    (12,8),
    (15,10)]

model = Model(GLPK.Optimizer)

@variable(model,x[T,J],Bin)
@objective(model, Max, sum( x[i,j] * b[j] for i=T,j=J))

@constraint(model, [j=J], sum( x[i,j] for i=T) <= 1)
@constraint(model, [i=T], sum( x[i,j] * a[j] for j=J) <= C[i])

for dep in dependencies
    @constraint(model, sum( x[i,dep[1]] for i=T)  <= sum( x[i,dep[2]] for i=T) )
end

@constraint(model, sum( x[i,11] + x[i,12] for i=T) <= 1)
@constraint(model, sum( x[i,2] + x[i,3] + x[i,4] + x[i,7] + x[i,8] for i=T ) >= 3)

optimize!(model)

println("solver status: ", termination_status(model))
println("solution status: ", primal_status(model))
println("object value: ", objective_value(model))

print("\n")
for i=T
    for j=J
        print(value(x[i,j]))
        print(" ")
    end
    print("\n")
end

solver status: OPTIMAL
solution status: FEASIBLE_POINT
object value: 244.0

0.0 1.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 1.0 0.0 1.0 
1.0 0.0 0.0 1.0 0.0 0.0 0.0 0.0 0.0 1.0 0.0 0.0 0.0 0.0 0.0 
0.0 0.0 1.0 0.0 0.0 1.0 0.0 0.0 1.0 0.0 0.0 0.0 0.0 1.0 0.0 


## Lagrangian Relaxation

We added one constraint to the objective function, by introducing an additional constant $u$
$$
Z(u) = max \sum_{i=1}^{3}\sum_{j=1}^{15} x_{ij}b_j + \sum_{i=1}^{3}u_i (C_i - \sum_{i=1}^3 x_{ij} a_j)
$$
and then minimized for $u$
$$
\min_{u} Z(u) 
$$


In [4]:
#Exercise 2
#Question 6/7   
# Lagrange Dual solution with gradient-descent -> min_u(max_x Z(u))
using GLPK, JuMP
J = 1:15                  # initiatives
T = 1:3                   # teams

b = [12, 24, 20, 28, 10, 11, 30, 27, 22, 25, 40, 38, 32, 34, 36]        # Strategic benefit
a = [14,  8,  7, 10, 13, 12,  9,  8,  7,  8, 12, 11, 10,  9, 10]               # Resource consumption
C = [28, 32, 35]    # Team capacities
u = [2.9,3.8,1.03]
x_solution = zeros((length(T),length(J)))

# Dependency pairs:
dependencies = [    # (i,j) means initiative i requires initiative j
    (2,1),
    (3,1),
    (4,1),
    (7,5),
    (8,5),
    (9,6),
    (10,6),
    (11,7),
    (12,8),
    (15,10)]

u = zeros(3)
bestUpperBound = Inf
bestu = copy(u)
maxIter = 500 

for k=1:maxIter
    # OPTIMIZATION MODEL
    model = Model(GLPK.Optimizer)

    @variable(model,1>=x[T,J]>=0)
    # @variable(model,x[T,J],Bin)

    @objective(model, Max, 
    
        sum( x[i,j] * b[j] for i=T,j=J)
        +
        sum( u[i] * (C[i] - sum( x[i,j] * a[j] for j=J) ) for i=T))

    @constraint(model, [j=J], sum( x[i,j] for i=T) <= 1)
    # @constraint(model, [i=T], sum( x[i,j] * a[j] for j=J) <= C[i])

    for dep in dependencies
        @constraint(model, sum( x[i,dep[1]] for i=T)  <= sum( x[i,dep[2]] for i=T) )
    end

    @constraint(model, sum( x[i,11] + x[i,12] for i=T) <= 1)
    @constraint(model, sum( x[i,2] + x[i,3] + x[i,4] + x[i,7] + x[i,8] for i=T ) == 3)

    optimize!(model)

    # UPDATING UPPER BOUND and u
    UB = objective_value(model)
    if UB < bestUpperBound
        bestUpperBound = UB
        bestu = copy(u)
        x_solution = value.(x)
    end

    # CALC NEW u 
    for i=T
        mu = 1/(2*k)    #0.05/sqrt(k)
        u[i] = max(u[i]  -  mu * ( C[i] - sum( value(x[i,j]) * a[j] for j=J) ), 0 )
    end
    # println("solver status: ", termination_status(model))
    # println("solution status: ", primal_status(model))
    if k % 25 == 0        
        println(
        "k = ", k,
        "   UB = ", round(UB,digits=2),
        "   best UB = ", round(bestUpperBound,digits=5),
        "   u = ", round.(u,digits=2))
    end
end

println("BestLagrangianUB = ", bestUpperBound)
println("Bestu = ", bestu)
println("x_solution = ")

display(x_solution)

k = 25   UB = 427.83   best UB = 307.0   u = [7.58, 3.21, 3.77]
k = 50   UB = 285.2   best UB = 275.71688   u = [2.79, 2.71, 2.92]
k = 75   UB = 278.51   best UB = 264.18141   u = [3.02, 2.53, 2.64]
k = 100   UB = 274.11   best UB = 263.75954   u = [2.93, 2.6, 2.67]
k = 125   UB = 267.2   best UB = 262.39093   u = [2.57, 2.64, 2.83]
k = 150   UB = 264.37   best UB = 261.23292   u = [2.7, 2.53, 2.69]
k = 175   UB = 261.85   best UB = 259.45821   u = [2.68, 2.67, 2.48]
k = 200   UB = 260.49   best UB = 259.45821   u = [2.66, 2.49, 2.59]
k = 225   UB = 262.68   best UB = 259.32057   u = [2.65, 2.51, 2.69]
k = 250   UB = 261.43   best UB = 258.65373   u = [2.64, 2.53, 2.68]
k = 275   UB = 258.89   best UB = 258.28763   u = [2.62, 2.68, 2.48]
k = 300   UB = 258.21   best UB = 257.53649   u = [2.61, 2.62, 2.5]
k = 325   UB = 261.42   best UB = 257.43396   u = [2.55, 2.57, 2.62]
k = 350   UB = 259.31   best UB = 257.43396   u = [2.65, 2.58, 2.52]
k = 375   UB = 260.02   best UB = 256.78447   

2-dimensional DenseAxisArray{Float64,2,...} with index sets:
    Dimension 1, 1:3
    Dimension 2, 1:15
And data, a 3×15 Matrix{Float64}:
 0.0       0.0       0.0       0.0       …  0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0       0.0       0.0       0.0          0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.333333  0.333333  0.333333  0.333333     0.0  0.0  0.0  1.0  1.0  1.0  0.0

## LP-Relaxation
letting the variables take real-values, not just integers and thereby receiving an upper bound, which is generally infeasible

In [3]:
#Exercise 2
#Question 8 - LP-Relaxation (positive real valued)
using GLPK, JuMP

J = 1:15                  # initiatives
T = 1:3                   # teams

b = [12, 24, 20, 28, 10, 11, 30, 27, 22, 25, 40, 38, 32, 34, 36]        # Strategic benefit
a = [14,  8,  7, 10, 13, 12,  9,  8,  7,  8, 12, 11, 10,  9, 10]               # Resource consumption
C = [28, 32, 35]    # Team capacities

# Dependency pairs:
dependencies = [    # (i,j) means initiative i requires initiative j
    (2,1),
    (3,1),
    (4,1),
    (7,5),
    (8,5),
    (9,6),
    (10,6),
    (11,7),
    (12,8),
    (15,10)]

model = Model(GLPK.Optimizer)

@variable(model,x[T,J]>=0)
@objective(model, Max, sum( x[i,j] * b[j] for i=T,j=J))

@constraint(model, [j=J], sum( x[i,j] for i=T) <= 1)
@constraint(model, [i=T], sum( x[i,j] * a[j] for j=J) <= C[i])

for dep in dependencies
    @constraint(model, sum( x[i,dep[1]] for i=T)  <= sum( x[i,dep[2]] for i=T) )
end

@constraint(model, sum( x[i,11] + x[i,12] for i=T) <= 1)
@constraint(model, sum( x[i,2] + x[i,3] + x[i,4] + x[i,7] + x[i,8] for i=T ) >= 3)

optimize!(model)

println("solver status: ", termination_status(model))
println("solution status: ", primal_status(model))
println("object value: ", objective_value(model))

print("\n")
for i=T
    for j=J
        print(value(x[i,j]))
        print(" ")
    end
    print("\n")
end

solver status: OPTIMAL
solution status: FEASIBLE_POINT
object value: 254.89189189189193

0.33333333333333315 0.33333333333333315 0.33333333333333315 0.0 0.7817047817047819 0.0 0.0 1.0000000000000004 0.0 0.0 6.504336911945362e-17 0.015561015561015568 0.0 0.0 0.0 
0.0 0.0 0.0 0.33333333333333315 0.0 0.5945945945945946 0.0 0.0 0.0 0.5945945945945946 0.0 0.9844389844389849 0.0 0.0 0.5945945945945946 
0.0 0.0 0.0 0.0 0.21829521829521836 0.0 1.0000000000000002 0.0 0.5945945945945946 0.0 0.0 0.0 1.0 1.0 0.0 
